# 02. Data Cleaning & Preprocessing
In this notebook, we clean anomalies, eliminate target leakage, engineer intuitive features, and create a reproducible train/test split for modeling.

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Paths
output_dir = '../results/02_data_preprocessing' if os.path.exists('../results') else 'results/02_data_preprocessing'
models_dir = '../models' if os.path.exists('../models') else 'models'
processed_dir = '../data/processed' if os.path.exists('../data') else 'data/processed'

os.makedirs(output_dir, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

# Load raw data
data_path = '../data/raw/hotel_bookings.csv' if os.path.exists('../data/raw') else 'data/raw/hotel_bookings.csv'
df = pd.read_csv(data_path)
print(f"Starting with {len(df):,} raw records.")

Starting with 119,390 raw records.


## 1. Remove Obvious Anomalies & Errors
- Filter out 180 ghost bookings (0 adults, 0 children, 0 babies).
- Filter out ADR pricing typos (ADR < 0 and the single 5400 extreme typo).

In [2]:
# 1. Filter ghost bookings
ghost_mask = (df['adults'] + df['children'].fillna(0) + df['babies']) == 0
df = df[~ghost_mask]

# 2. Filter ADR errors
df = df[(df['adr'] >= 0) & (df['adr'] <= 5000)]

print(f"Clean records remaining: {len(df):,}")

Clean records remaining: 119,208


## 2. Eliminate Target Leakage & Engineer Features
- **Drop `reservation_status` and `reservation_status_date`**: These columns record what happened after the booking finished (e.g. 'Check-Out' or 'Canceled'). Using them would cause 100% target leakage.
- **Engineer new features**: Total stay nights, total guests, special requests flag, booking changes flag, and room type change flag.

In [3]:
# Drop target leakage columns
df = df.drop(columns=['reservation_status', 'reservation_status_date'])

# Feature Engineering
df['total_stay_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests'] = df['adults'] + df['children'].fillna(0) + df['babies']
df['has_special_requests'] = (df['total_of_special_requests'] > 0).astype(int)
df['has_booking_changes'] = (df['booking_changes'] > 0).astype(int)
df['room_type_changed'] = (df['reserved_room_type'] != df['assigned_room_type']).astype(int)

# Simple missing value imputation
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna(0)
df['company'] = df['company'].fillna(0)

print(f"Missing values remaining: {df.isnull().sum().sum()}")

Missing values remaining: 0


## 3. Stratified Train / Test Split
We split the data into 80% for training and 20% for testing, preserving the cancellation proportion.

In [4]:
X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training instances: {X_train.shape[0]:,} (Cancel rate: {y_train.mean():.2%})")
print(f"Testing instances:  {X_test.shape[0]:,} (Cancel rate: {y_test.mean():.2%})")

Training instances: 95,366 (Cancel rate: 37.08%)
Testing instances:  23,842 (Cancel rate: 37.08%)


## 4. Feature Encoding & Scaling
We build a Scikit-Learn `ColumnTransformer` to standardize numeric features and one-hot encode categorical features. We fit **only** on the training data.

In [5]:
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

# Fit on training data only
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f"Transformed feature matrix: {X_train_processed.shape[1]} features.")

C:\Users\abdel\AppData\Local\Temp\ipykernel_24968\2708431662.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()


Transformed feature matrix: 255 features.


## 5. Save Processed Datasets & Preprocessor Model

In [6]:
# Save processed datasets
X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)

X_train_df.to_parquet(f'{processed_dir}/X_train.parquet')
X_test_df.to_parquet(f'{processed_dir}/X_test.parquet')
pd.DataFrame(y_train).to_csv(f'{processed_dir}/y_train.csv', index=False)
pd.DataFrame(y_test).to_csv(f'{processed_dir}/y_test.csv', index=False)

# Save preprocessor model to models/
joblib.dump(preprocessor, f'{models_dir}/preprocessor.joblib')

# Save summary report
summary_table = pd.DataFrame({
    'Metric': ['Initial Raw Rows', 'Cleaned Records', 'Training Rows (80%)', 'Testing Rows (20%)', 'Total Predictors'],
    'Value': [119390, len(df), len(X_train), len(X_test), len(feature_names)]
})
summary_table.to_csv(f'{output_dir}/preprocessing_summary.csv', index=False)
print("Data preprocessing complete and artifacts saved!")
summary_table

Data preprocessing complete and artifacts saved!


,Metric,Value
0,Initial Raw Rows,119390
1,Cleaned Records,119208
2,Training Rows (80%),95366
3,Testing Rows (20%),23842
4,Total Predictors,255
